# Disc-and-cup benchmark — analysis

The generated pages hold the facts: [how it is run](../docs/benchmarks/disc-docs.md) and
[what came out](../docs/benchmarks/disc-results.md). This notebook is where the judgement goes.

The benchmark's question is **model against the outline an ophthalmologist drew**, so the model is
the primary index everywhere below, and the dataset, the reader and the structure sit inside it.
Model-against-model numbers are here to raise suspicions about the outlines, never to crown
anything.

Every measurement is in the **native frame** — the full-resolution square the store built, which is
where the expert drew — so a pixel here is a pixel of the original photograph.

Run it after `python -m benchmarks --benchmark disc`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def repository() -> Path:
    """The repository, found rather than assumed.

    A notebook is opened from wherever its reader happens to be — the repository root, the
    notebooks directory, an editor's workspace — and a relative path that is right in one of those
    is wrong in the others.
    """
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(
        "this notebook has to be run from inside the fundus-atlas repository; "
        f"nothing above {Path.cwd()} looks like it"
    )


ROOT = repository()
sys.path.insert(0, str(ROOT / "src"))
RESULTS = ROOT / "results" / "disc"
STORE = ROOT / ".atlas_data"
KEPT_MASKS = ROOT / ".atlas_runs" / "disc"

# The run's own measurements, reused here so that the readers' ceiling in section 5 is computed
# exactly the way the models were scored rather than by a second implementation of Dice.
from benchmarks.metrics import disc as measurements  # noqa: E402
from datasets.utils import contours as contour_files  # noqa: E402

STRUCTURES = ["disc", "cup"]


def per_image() -> pd.DataFrame:
    """Every model's answer about every photograph, one row per reader it was scored against."""
    frames = []
    for path in sorted(RESULTS.glob("*/*.csv")):
        frame = pd.read_csv(path)
        frame["model"] = path.parent.name
        frame["dataset"] = path.stem
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(
            f"no results in {RESULTS} — run `python -m benchmarks --benchmark disc` first"
        )
    return pd.concat(frames, ignore_index=True)


def summaries() -> pd.DataFrame:
    """What each run recorded about itself: how much it covered, and how long it took."""
    rows = []
    for path in sorted(RESULTS.glob("*/*.json")):
        record = json.loads(path.read_text())
        kept = record["summary"]
        rows.append({
            "model": record["model"],
            "dataset": record["dataset"],
            "photographs": kept.get("photographs"),
            "outlines": kept.get("outlines"),
            "failed": kept.get("failed"),
            "processed": kept.get("processed"),
            "total": kept.get("total"),
            "complete": kept.get("complete"),
            "seconds_per_photograph": kept.get("seconds_per_photograph"),
            "device": kept.get("device"),
            "structures": ", ".join(kept.get("structures", [])),
        })
    return pd.DataFrame(rows).set_index(["model", "dataset"]).sort_index()


rows = per_image()
overview = summaries()
MODELS = sorted(rows["model"].unique())
DATASETS = sorted(rows["dataset"].unique())
GRADED = rows[rows["outcome"] == "graded"]

print(f"{len(rows):,} scored outlines · {len(MODELS)} models · {len(DATASETS)} datasets")
print("models:", ", ".join(MODELS))
print("datasets:", ", ".join(DATASETS))

## 1. Coverage first

A model that produced no outline has not got one wrong. Coverage here is simply whether the model
answered at all: these are segmentation networks, and unlike the quality graders none of them has a
way of declining, so anything other than `graded` is a failure with a reason attached.

The counts are **outlines, not photographs**: a photograph five ophthalmologists drew on makes five
rows, because the model is scored against each of them separately.

In [ ]:
coverage = (rows.pivot_table(index=["model", "dataset"], columns="outcome", values="key",
                             aggfunc="count").fillna(0).astype(int))
coverage.join(overview[["photographs", "processed", "total", "complete", "structures"]])

In [ ]:
failures = rows[rows["outcome"] != "graded"]
if failures.empty:
    print("every photograph was outlined by every model")
else:
    print(failures.groupby(["model", "dataset"])["note"].value_counts().to_string())

### 1.1 Which structures each model is in the tables for

A model that finds only the disc is **absent** from every cup table below rather than scored zero in
it. That is why the cup columns have fewer models in them than the disc columns, and it is not a
gap in the run.

In [ ]:
found = pd.DataFrame(
    {structure: [GRADED.loc[GRADED["model"] == model, f"{structure}_dice"].notna().any()
                 for model in MODELS] for structure in STRUCTURES},
    index=MODELS,
)
found.replace({True: "scored", False: "—"})

## 2. Overlap, briefly

Dice is the familiar number and the least informative one here: it says how much two outlines share,
not where they differ. Two models a hundredth apart on Dice can put the boundary in quite different
places, and it is the boundary that every clinical number is computed from. So the overlap comes
first and short, and sections 3 and 4 say what it is hiding.

A Dice score of 1 is perfect agreement, 0 no overlap at all.

In [ ]:
pooled = GRADED.groupby("model")[["disc_dice", "cup_dice"]].agg(["mean", "median", "std", "count"])
pooled.round(3)

### 2.1 The same numbers, per dataset

Pooling hides the datasets. Each of these is a different camera, a different population and a
different set of hands drawing the outlines.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for axis, structure in zip(axes, STRUCTURES):
    table = GRADED.pivot_table(index="model", columns="dataset", values=f"{structure}_dice",
                               aggfunc="mean")
    drawn = axis.imshow(table.values, cmap="viridis", vmin=0.5, vmax=1.0, aspect="auto")
    axis.set_xticks(range(len(table.columns)), table.columns, rotation=30, ha="right")
    axis.set_yticks(range(len(table.index)), table.index)
    axis.set_title(f"{structure} — mean Dice")
    for y in range(table.shape[0]):
        for x in range(table.shape[1]):
            value = table.values[y, x]
            if not np.isnan(value):
                axis.text(x, y, f"{value:.3f}", ha="center", va="center",
                          color="white" if value < 0.85 else "black", fontsize=9)
    fig.colorbar(drawn, ax=axis, shrink=0.85)
fig.suptitle("Overlap with the expert's outline, by model and dataset")
fig.tight_layout()

### 2.2 The distribution behind each mean

A mean Dice of 0.93 can be every photograph at 0.93, or most of them at 0.96 with a tail of
failures. The tail is what a study feels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for axis, structure in zip(axes, STRUCTURES):
    series = [GRADED.loc[GRADED["model"] == model, f"{structure}_dice"].dropna() for model in MODELS]
    present = [(model, values) for model, values in zip(MODELS, series) if len(values)]
    axis.boxplot([values for _, values in present], tick_labels=[model for model, _ in present],
                 vert=True, showfliers=True, flierprops={"markersize": 2, "alpha": 0.3})
    axis.set_title(f"{structure} Dice, every outline")
    axis.set_ylim(0, 1.02)
    axis.tick_params(axis="x", rotation=30)
    axis.grid(axis="y", alpha=0.3)
fig.tight_layout()

In [ ]:
# How often a model is not merely imprecise but somewhere else entirely.
lost = {
    structure: {
        model: float((GRADED.loc[GRADED["model"] == model, f"{structure}_dice"].dropna() < 0.5).mean())
        for model in MODELS
        if GRADED.loc[GRADED["model"] == model, f"{structure}_dice"].notna().any()
    }
    for structure in STRUCTURES
}
pd.DataFrame(lost).rename(columns=lambda name: f"share of {name} outlines under Dice 0.5").round(4)

## 3. Where the boundary actually sits

This is the section Dice cannot answer. Three questions, in order of how much they matter to a
measurement:

- **Is the structure in the right place?** — the distance between the two centres, given in the
  expert's own disc diameters so that photographs of different sizes can be read together.
- **Is it the right size?** — the signed error in width, height and equivalent radius. Signed,
  because a model that always draws the cup too large is a different problem from one that
  scatters, and the average of an absolute error hides which of the two you have.
- **Does the size error have a direction?** — a model short vertically and right horizontally is
  making an elliptical error, and the vertical cup-to-disc ratio inherits it.

In [ ]:
placement = GRADED.groupby("model")[["disc_centre_offset", "disc_centre_offset_diameters",
                                     "cup_centre_offset", "cup_centre_offset_diameters"]].median()
placement.round(4)

In [ ]:
fig, axis = plt.subplots(figsize=(11, 4))
present = [(model, GRADED.loc[GRADED["model"] == model, "disc_centre_offset_diameters"].dropna())
           for model in MODELS]
present = [(model, values) for model, values in present if len(values)]
axis.boxplot([values for _, values in present], tick_labels=[model for model, _ in present],
             showfliers=False)
axis.axhline(0.1, color="crimson", linestyle="--", linewidth=1,
             label="a tenth of a disc diameter")
axis.set_ylabel("disc centre offset, in the expert's disc diameters")
axis.set_title("Is the disc in the right place?")
axis.legend()
axis.grid(axis="y", alpha=0.3)
fig.tight_layout()

### 3.1 Signed size errors

Positive means the model drew the structure **larger** than the ophthalmologist did.

In [ ]:
size = GRADED.groupby("model")[[f"{s}_{m}_error" for s in STRUCTURES
                                for m in ("width", "height", "radius")]].agg(["mean", "std"])
size.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for axis, structure in zip(axes, STRUCTURES):
    for model in MODELS:
        subset = GRADED[GRADED["model"] == model]
        wide, tall = subset[f"{structure}_width_error"], subset[f"{structure}_height_error"]
        if wide.notna().any():
            axis.scatter(wide.mean(), tall.mean(), s=90, label=model)
            axis.errorbar(wide.mean(), tall.mean(), xerr=wide.sem(), yerr=tall.sem(), alpha=0.5)
    axis.axhline(0, color="grey", linewidth=1)
    axis.axvline(0, color="grey", linewidth=1)
    axis.set_xlabel("width error, native pixels (+ = model wider)")
    axis.set_ylabel("height error (+ = model taller)")
    axis.set_title(f"{structure}: the shape of the error")
    axis.grid(alpha=0.3)
axes[0].legend(fontsize=8)
fig.tight_layout()

### 3.2 Per dataset, because the bias is not one number

A model can be right on one camera and systematically large on another. Where that happens, the
pooled bias above is an average of two different behaviours.

In [ ]:
GRADED.pivot_table(index="model", columns="dataset",
                   values=["disc_radius_error", "cup_radius_error"], aggfunc="mean").round(1)

## 4. The number that leaves the building

The **vertical cup-to-disc ratio** — the height of the cup over the height of the disc — is what a
referral decision rests on. It is a quotient of two boundaries, so a model can overlap both
structures well and still get the ratio wrong: the two errors need not cancel.

The error is signed. **Positive means the model reads the ratio higher than the ophthalmologist
did**, which is the direction that sends people to a clinic who were not sent by the expert.

In [ ]:
ratio = GRADED.dropna(subset=["cup_vertical_ratio_error"])
fig, axes = plt.subplots(1, max(1, ratio["model"].nunique()),
                         figsize=(4.2 * max(1, ratio["model"].nunique()), 4.2), squeeze=False)
for axis, model in zip(axes[0], sorted(ratio["model"].unique())):
    subset = ratio[ratio["model"] == model]
    for dataset in sorted(subset["dataset"].unique()):
        here = subset[subset["dataset"] == dataset]
        axis.scatter(here["truth_vertical_ratio"], here["said_vertical_ratio"], s=6, alpha=0.35,
                     label=dataset)
    axis.plot([0, 1], [0, 1], color="black", linewidth=1)
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.set_xlabel("the expert's ratio")
    axis.set_ylabel("the model's ratio")
    axis.set_title(model, fontsize=10)
    axis.legend(fontsize=7, markerscale=2)
fig.suptitle("Vertical cup-to-disc ratio: model against the ophthalmologist, per photograph")
fig.tight_layout()

In [ ]:
crossing = 0.6  # a threshold in common clinical use; nothing here endorses it


def referral(subset: pd.DataFrame) -> pd.Series:
    said, truth = subset["said_vertical_ratio"], subset["truth_vertical_ratio"]
    return pd.Series({
        "bias": subset["cup_vertical_ratio_error"].mean(),
        "spread": subset["cup_vertical_ratio_error"].std(),
        "within 0.05": float(subset["cup_vertical_ratio_error"].abs().le(0.05).mean()),
        f"over {crossing} when the expert was not": float(
            ((said >= crossing) & (truth < crossing)).mean()),
        f"under {crossing} when the expert was not": float(
            ((said < crossing) & (truth >= crossing)).mean()),
    })


ratio.groupby(["model", "dataset"]).apply(referral, include_groups=False).round(3)

### 4.1 The area ratio, for the projects that compute it that way

Some pipelines divide the cup's **area** by the disc's rather than their heights. It is a different
number and it fails differently.

In [ ]:
GRADED.groupby("model")[["cup_area_ratio_error", "cup_outside_its_disc"]].agg(
    ["mean", "std"]).round(4)

## 5. The readers' own disagreement is the ceiling

A model cannot be said to be wrong by less than the ophthalmologists are wrong about each other.
[Chákṣu](../docs/datasets/chaksu.md) publishes five readers' outlines and
[PAPILA](../docs/datasets/papila.md) two, so on those two datasets the ceiling can be measured:
score every reader against every other reader, exactly as the models were scored, and read the
models against that band.

Where a dataset publishes one reader, **no ceiling can be computed** — that is stated rather than
left to be inferred.

The pairs are computed on a sample of photographs rather than all of them: each pair means
rasterising two outlines at full resolution, and the band is stable long before the sample runs
out. The sample is fixed by a seed so the numbers do not move between runs.

In [ ]:
SAMPLE = 60
SEED = 0


def readers_of(dataset: str) -> dict:
    """Every outline drawn on each photograph of one dataset, in the native frame."""
    drawn = {}
    for path in sorted((STORE / dataset / "native" / "contours").glob("*.csv")):
        drawn[path.stem] = contour_files.read(path)
    return drawn


def sides_of(dataset: str) -> dict:
    manifest = pd.read_csv(STORE / dataset / "manifest.csv")
    return dict(zip(manifest["key"], manifest["crop_side"]))


def ceiling(dataset: str) -> pd.DataFrame:
    """Reader against reader, on the measurements the models are scored on."""
    drawn, sides = readers_of(dataset), sides_of(dataset)
    keys = sorted(drawn)
    chosen = np.random.default_rng(SEED).choice(keys, size=min(SAMPLE, len(keys)), replace=False)
    found = []
    for key in chosen:
        side = int(sides[key])
        readers = sorted({reader for _, reader in drawn[key]})
        if len(readers) < 2:
            continue
        masks = {
            (structure, reader): measurements.mask_of(nodes, (side, side))
            for (structure, reader), nodes in drawn[key].items()
        }
        for one in range(len(readers)):
            for other in range(one + 1, len(readers)):
                first, second = readers[one], readers[other]
                said = {s: masks[(s, first)] for s in STRUCTURES if (s, first) in masks}
                truth = {s: masks[(s, second)] for s in STRUCTURES if (s, second) in masks}
                if not said or not truth:
                    continue
                found.append({"key": key, "pair": f"{first}~{second}",
                              **measurements.measure(said, truth)})
    return pd.DataFrame(found)


ceilings = {dataset: ceiling(dataset) for dataset in DATASETS}
for dataset, table in ceilings.items():
    if table.empty:
        print(f"{dataset}: one reader — no ceiling can be computed")
    else:
        print(f"{dataset}: {table['pair'].nunique()} reader pairs over {table['key'].nunique()} "
              f"photographs")

In [ ]:
band = []
for dataset, table in ceilings.items():
    if table.empty:
        continue
    for structure in STRUCTURES:
        column = f"{structure}_dice"
        if column in table and table[column].notna().any():
            band.append({"dataset": dataset, "structure": structure, "who": "reader vs reader",
                         "dice": table[column].mean(),
                         "ratio error": table["cup_vertical_ratio_error"].abs().mean()
                         if "cup_vertical_ratio_error" in table else np.nan})
for (model, dataset), subset in GRADED.groupby(["model", "dataset"]):
    for structure in STRUCTURES:
        if subset[f"{structure}_dice"].notna().any():
            band.append({"dataset": dataset, "structure": structure, "who": model,
                         "dice": subset[f"{structure}_dice"].mean(),
                         "ratio error": subset["cup_vertical_ratio_error"].abs().mean()})
pd.DataFrame(band).pivot_table(index=["structure", "who"], columns="dataset",
                               values="dice").round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for axis, structure in zip(axes, STRUCTURES):
    for dataset, table in ceilings.items():
        if table.empty or f"{structure}_dice" not in table:
            continue
        readers = table[f"{structure}_dice"].dropna()
        if readers.empty:
            continue
        axis.axhspan(readers.quantile(0.25), readers.quantile(0.75), alpha=0.15,
                     label=f"{dataset}: readers' middle half")
    models = GRADED.dropna(subset=[f"{structure}_dice"])
    for offset, model in enumerate(sorted(models["model"].unique())):
        by_dataset = models[models["model"] == model].groupby("dataset")[f"{structure}_dice"].mean()
        axis.scatter([offset] * len(by_dataset), by_dataset.values, s=60)
        for dataset, value in by_dataset.items():
            axis.annotate(dataset[:4], (offset, value), fontsize=7,
                          textcoords="offset points", xytext=(6, 0))
    names = sorted(models["model"].unique())
    axis.set_xticks(range(len(names)), names, rotation=30, ha="right")
    axis.set_title(f"{structure}: models against the readers' own agreement")
    axis.set_ylabel("Dice")
    axis.legend(fontsize=7)
    axis.grid(axis="y", alpha=0.3)
fig.tight_layout()

## 6. Model against model

Two models agreeing is not evidence that either agrees with an expert — and it is weaker still when
the two were fitted on the same images. Three of the models here name **REFUGE** in their training
data, so their agreement with each other is close to one piece of evidence rather than three.

The comparison below is on the number the models actually deliver, the vertical cup-to-disc ratio,
on the photographs both of them outlined.

In [ ]:
ratios = GRADED.drop_duplicates(["model", "dataset", "key"]).pivot_table(
    index=["dataset", "key"], columns="model", values="said_vertical_ratio")
present = [model for model in MODELS if model in ratios.columns]
difference = pd.DataFrame(index=present, columns=present, dtype=float)
for one in present:
    for other in present:
        both = ratios[[one, other]].dropna()
        difference.loc[one, other] = (both[one] - both[other]).abs().mean() if len(both) else np.nan

fig, axis = plt.subplots(figsize=(6.5, 5.2))
drawn = axis.imshow(difference.values.astype(float), cmap="magma_r")
axis.set_xticks(range(len(present)), present, rotation=30, ha="right")
axis.set_yticks(range(len(present)), present)
for y in range(len(present)):
    for x in range(len(present)):
        value = difference.values[y, x]
        if not np.isnan(float(value)):
            axis.text(x, y, f"{float(value):.3f}", ha="center", va="center", fontsize=9,
                      color="white" if float(value) > difference.values.astype(float).max() / 2
                      else "black")
axis.set_title("Mean difference in the cup-to-disc ratio\nbetween two models, same photographs")
fig.colorbar(drawn, ax=axis, shrink=0.8)
fig.tight_layout()

## 7. The hard cases

An outline is a picture, and this is the benchmark where the pictures matter most. Below, for each
dataset in turn, are the photographs the models found hardest: the expert's outline in white, the
model's in colour, drawn over the photograph itself.

They are taken **from each dataset in turn** rather than from the pile — a grid drawn from whichever
dataset happens to produce the most failures is a picture of that dataset, not of the benchmark.

In [ ]:
from PIL import Image  # noqa: E402

SHOWN = 4  # photographs per dataset, each drawn once per model that outlined it


def photograph(dataset: str, key: str, size: int = 512) -> np.ndarray:
    with Image.open(STORE / dataset / str(size) / "images" / f"{key}.png") as image:
        return np.asarray(image.convert("RGB"))


def kept_mask(model: str, dataset: str, key: str, structure: str, size: int = 512):
    """The mask the run kept, brought down to the grid the photograph is drawn at."""
    path = KEPT_MASKS / model / dataset / f"{key}-{structure}.png"
    if not path.exists():
        return None
    with Image.open(path) as mask:
        return np.asarray(mask.resize((size, size), Image.Resampling.NEAREST)) > 0


def expert_outline(dataset: str, key: str, structure: str, size: int = 512):
    path = STORE / dataset / str(size) / "contours" / f"{key}.csv"
    if not path.exists():
        return []
    drawn = contour_files.read(path)
    return [nodes for (found, _), nodes in drawn.items() if found == structure]


for dataset in DATASETS:
    here = GRADED[GRADED["dataset"] == dataset]
    hardest = (here.groupby("key")["disc_dice"].mean().dropna().nsmallest(SHOWN).index.tolist())
    models = sorted(here["model"].unique())
    if not hardest:
        continue
    fig, axes = plt.subplots(len(hardest), len(models),
                             figsize=(3.1 * len(models), 3.1 * len(hardest)), squeeze=False)
    for y, key in enumerate(hardest):
        for x, model in enumerate(models):
            axis = axes[y][x]
            axis.imshow(photograph(dataset, key))
            for structure, colour in (("disc", "#ff4d4d"), ("cup", "#4da6ff")):
                mask = kept_mask(model, dataset, key, structure)
                if mask is not None and mask.any():
                    axis.contour(mask, levels=[0.5], colors=[colour], linewidths=1.4)
                for nodes in expert_outline(dataset, key, structure):
                    axis.plot(nodes[:, 0], nodes[:, 1], color="white", linewidth=1.0, alpha=0.9)
            scored = here[(here["key"] == key) & (here["model"] == model)]["disc_dice"].mean()
            axis.set_title(f"{model}\n{key} · disc Dice {scored:.2f}", fontsize=8)
            axis.axis("off")
    fig.suptitle(f"{dataset}: the four photographs with the lowest mean disc overlap\n"
                 f"white = the ophthalmologist · red = the model's disc · blue = its cup",
                 fontsize=10)
    fig.tight_layout()
    plt.show()

## 8. What each model costs to run

A model twice as slow for a hundredth of Dice is a different proposition at fifty thousand
photographs than at fifty, and that trade is invisible in an overlap table. The seconds below are
what the run recorded: **the model's own call, per photograph, once its weights were loaded**.

They measure this machine as much as the model. A different device, and every number changes.

In [ ]:
overview[["seconds_per_photograph", "device", "photographs"]].round(3)

In [ ]:
speed = overview.groupby("model")["seconds_per_photograph"].mean()
quality = GRADED.groupby("model")["disc_dice"].mean()
fig, axis = plt.subplots(figsize=(7, 4.5))
for model in MODELS:
    if model in speed and model in quality:
        axis.scatter(speed[model], quality[model], s=90)
        axis.annotate(model, (speed[model], quality[model]), fontsize=8,
                      textcoords="offset points", xytext=(7, 0))
axis.set_xlabel("seconds per photograph")
axis.set_ylabel("mean disc Dice")
axis.set_title("What the overlap costs")
axis.grid(alpha=0.3)
fig.tight_layout()

## 9. What this analysis cannot say

- **`unknown` is not `out-of-sample`.** Two of these models publish no training list at all, so
  nothing here can be called a clean result for them. A model that trained on a dataset it is being
  scored on will look better than it is, and the marks in
  [docs/BENCHMARKS.md](../docs/BENCHMARKS.md) are where to check which is which.
- **Three of the models trained on REFUGE**, which none of these datasets is — but which also means
  their agreement with each other is not three independent pieces of evidence.
- **The resampling path moves the numbers.** Every model here emits probabilities that are carried
  to the native frame and thresholded there; a model whose masks had been thresholded first would
  score differently, and not by a constant.
- **Two of the models emit exclusive classes** — a disc ring and the cup inside it — and the
  adapters put the ring and the cup back together, because an ophthalmologist's disc contour
  contains the cup. That is a decision this repository made, and it is visible in the code rather
  than in the model's own documentation.
- **A reader is not the truth.** Section 5 measures how far the ophthalmologists are from each
  other; where a model sits inside that band, it is not distinguishable from a reader, and no
  ranking below that resolution means anything.
- **The cup is harder than the disc for people too.** A cup boundary is a judgement about where the
  surface of the nerve head begins to slope, and the readers' own disagreement on it is wide. Treat
  every cup number here as resting on a softer reference than the disc numbers do.
- **Nothing here is a claim about glaucoma.** Every number is agreement with an outline somebody
  drew, and a cup-to-disc ratio is one input to a diagnosis rather than the diagnosis.